In [20]:
!pip install -q google-genai gradio pandas plotly

import os, re, glob, inspect
import pandas as pd
import gradio as gr
import plotly.graph_objects as go
from google import genai
from google.genai import types


In [21]:
#=========================================================
# 【區塊 0：參數設定】
# CSV 裡「沒有」的欄位（油價、電價、油耗、電耗）集中放這裡，之後只需要改這一區
# =========================================================
GAS_PRICE_PER_L    = 31.0   # 汽油 NT$/公升（沿用原程式）
ELEC_PRICE_PER_KWH = 4.5    # 電價 NT$/度（沿用原程式）
EV_KM_PER_KWH      = 5.8    # bZ4X 每度電可跑公里數（沿用原程式）
FUEL_KM_PER_L = {"汽油": 12.0, "油電混合": 20.0}  # 各動力型式平均油耗 km/L。汽油沿用原程式；油電混合是估計值，請依官方規格調整
FUEL_KM_PER_L_BY_MODEL = {}  # 想精準到單一車款時在這裡覆寫，例如 {"ALTIS": 14.5, "RAV4 HV": 21.0}

A_CURRENCY_TO_NTD = 1.0     # CSV 金額是「A國幣」(非台幣)。填 1 A國幣 = 幾元台幣；目前先以 1:1 當 NT$ 使用
YEARS = 5                   # TCO 計算年數
MAINT_AGE_BAND = (1, 5)     # 保養費只取「車齡 1~5 年」的車主，口徑與 5 年 TCO 一致（避免老車/新車混在一起比）
MIN_SAMPLE = 30             # 某車款在上述車齡區間內少於此人數 → 改用「同動力型式（汽油/油電混合）車款」的平均
EV_MODEL = "bZ4X"

MAIN_CSV_PATH = None        # CSV 不在目前資料夾時，填完整路徑；None = 自動搜尋
SUPP_CSV_PATH = None

GEMINI_API_KEY = "AQ.Ab8RN6KXPn6Asww76XRvvu6PjbtZuiZnaqA91U9275wf0QyT-w"   # 👉 請在此填寫金鑰（若 Colab 左側「密鑰」有名為 api 的 secret，會優先使用）
GEMINI_MODELS = ["gemini-3.6-flash"]  # 依序嘗試；舊的 gemini-1.5-flash 已下線

In [22]:
#---------------------------------------------------------
# 【區塊 1：讀取兩份 CSV 檔案】
# 兩份檔案都是 Big5 編碼（不是 UTF-8），欄位名稱還夾有換行，所以要自動判斷編碼 + 正規化欄名
# ---------------------------------------------------------
COLUMN_KEYS = {
    "sex": "性別", "age": "年齡", "city": "居住縣市",
    "visits_1y": "近一年回廠次數",
    "svc_amt_1y": "近一年定保消費金額",
    "repair_cnt_1y": "近一年維修次數",
    "model": "現保有車款",
    "deliver": "現保有車交車日期",
}

def _find_csv(path, must_have, must_not=()):
    if path:
        return path
    hits = []
    for d in (".", "/content"):
        for p in glob.glob(os.path.join(d, "*.csv")):
            name = os.path.basename(p)
            if all(k in name for k in must_have) and not any(k in name for k in must_not):
                hits.append(p)
    if not hits:
        raise FileNotFoundError(f"找不到檔名含 {must_have} 的 CSV，請先上傳到 Colab，或在區塊 0 填入 *_CSV_PATH")
    return max(hits, key=os.path.getsize)

def _read_csv_auto(path):
    for enc in ("utf-8-sig", "big5", "cp950"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"無法判斷 {path} 的編碼")

def _load_cdp(path):
    raw = _read_csv_auto(path)
    raw.columns = [re.sub(r"\s+", "", str(c)) for c in raw.columns]   # 去掉欄名裡的換行/空白
    out = pd.DataFrame()
    for key, kw in COLUMN_KEYS.items():
        hits = [c for c in raw.columns if kw in c]
        if len(hits) != 1:
            raise ValueError(f"{os.path.basename(path)}：找不到唯一欄位「{kw}」，目前欄位：{list(raw.columns)}")
        out[key] = raw[hits[0]]
    for c in ["age", "visits_1y", "svc_amt_1y", "repair_cnt_1y"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out["deliver"] = pd.to_datetime(out["deliver"], errors="coerce")
    out["model"] = out["model"].astype(str).str.strip()
    out["age"] = out["age"].where(out["age"].between(18, 100))        # 0 = 未提供、>100 = 異常值，一律視為缺值
    out["svc_amt_1y"] = out["svc_amt_1y"] * A_CURRENCY_TO_NTD         # A國幣 → 台幣
    return out

df_main = _load_cdp(_find_csv(MAIN_CSV_PATH, ("CDP",), ("補充",)))   # 主檔：各車款車主回廠資料
df_supp = _load_cdp(_find_csv(SUPP_CSV_PATH, ("補充",)))              # 補充檔：bZ4X 車主資料
print(f"✅ CSV 資料庫讀取成功！主檔 {len(df_main):,} 筆、bZ4X 補充檔 {len(df_supp):,} 筆")  # 依序嘗試；舊的 gemini-1.5-flash 已下線

✅ CSV 資料庫讀取成功！主檔 1,048,575 筆、bZ4X 補充檔 3,037 筆


In [23]:
# ---------------------------------------------------------
# 【區塊 2：從 CSV 統計各車款保養費 + 5年 TCO 養車費用預估】
# ---------------------------------------------------------
# 主檔剛好停在 Excel 列數上限(1,048,575)，bZ4X 不完整 → bZ4X 一律用補充檔，其他車款用主檔
df_all = pd.concat([df_main[df_main["model"] != EV_MODEL], df_supp], ignore_index=True)
SNAPSHOT = df_all["deliver"].max()                                    # 資料截止日（以最晚交車日代表）
df_all["years"] = (SNAPSHOT - df_all["deliver"]).dt.days / 365.25     # 截至資料截止日的車齡

lo, hi = MAINT_AGE_BAND

def _powertrain(model):
    m = model.upper()
    if model == EV_MODEL:
        return "純電"
    return "油電混合" if ("HV" in m or m.endswith("-H") or m.startswith("PRIUS")) else "汽油"

def _robust_mean(s, q=0.99):
    s = s.dropna()
    return float(s.clip(upper=s.quantile(q)).mean()) if len(s) else float("nan")   # 蓋掉最高 1% 極端值再平均

df_all["powertrain"] = df_all["model"].map(_powertrain)
_band_ice = df_all[df_all["years"].between(lo, hi) & (df_all["model"] != EV_MODEL)]
POOLED_MAINT = {pt: _robust_mean(g["svc_amt_1y"]) for pt, g in _band_ice.groupby("powertrain")}   # 樣本不足時的備援：同動力型式平均

def _build_model_stats():
    rows = []
    for model, g in df_all.groupby("model"):
        band = g[g["years"].between(lo, hi)]
        enough = len(band) >= MIN_SAMPLE
        rows.append({
            "model": model,
            "powertrain": g["powertrain"].iloc[0],
            "owners": len(g),
            "median_years": g["years"].median(),
            "n_band": len(band),
            "maint_per_year": _robust_mean(band["svc_amt_1y"]) if enough else POOLED_MAINT[g["powertrain"].iloc[0]],
            "maint_fallback": not enough,
            "visits_band": _robust_mean(band["visits_1y"]) if enough else float("nan"),
            "repair_band": _robust_mean(band["repair_cnt_1y"]) if enough else float("nan"),
        })
    return pd.DataFrame(rows).set_index("model")

MODEL_STATS = _build_model_stats()
EV_STATS  = MODEL_STATS.loc[EV_MODEL]
ICE_STATS = MODEL_STATS.drop(index=EV_MODEL)
CAR_CHOICES = ICE_STATS[ICE_STATS["owners"] >= MIN_SAMPLE].sort_values("owners", ascending=False).index.tolist()

def calculate_tco(current_car_type, annual_km, years=YEARS):
    s = ICE_STATS.loc[current_car_type]
    km_per_l = FUEL_KM_PER_L_BY_MODEL.get(current_car_type, FUEL_KM_PER_L[s["powertrain"]])

    ice_energy_cost = (annual_km / km_per_l) * GAS_PRICE_PER_L * years
    ev_energy_cost  = (annual_km / EV_KM_PER_KWH) * ELEC_PRICE_PER_KWH * years
    ice_maint_cost  = s["maint_per_year"] * years          # ← 來自 CSV
    ev_maint_cost   = EV_STATS["maint_per_year"] * years   # ← 來自 CSV（bZ4X 補充檔）

    total_ice = ice_energy_cost + ice_maint_cost
    total_ev  = ev_energy_cost + ev_maint_cost
    saved_money = total_ice - total_ev
    detail = {
        "km_per_l": km_per_l, "powertrain": s["powertrain"],
        "ice_maint_yr": s["maint_per_year"], "ice_n": int(s["n_band"]), "ice_fallback": bool(s["maint_fallback"]),
        "ev_maint_yr": EV_STATS["maint_per_year"], "ev_n": int(EV_STATS["n_band"]),
    }
    return total_ice, total_ev, saved_money, detail



In [24]:
# ---------------------------------------------------------
# 【區塊 3：Gemini API 邏輯】
# 原本把 100 萬列 CSV 整份塞進 prompt 會爆 token，改成餵「CSV 統計摘要」
# ---------------------------------------------------------
def _load_api_key():
    try:
        from google.colab import userdata
        return userdata.get("api") or GEMINI_API_KEY
    except Exception:
        return GEMINI_API_KEY

API_KEY = _load_api_key()
client = genai.Client(api_key=API_KEY)

def _fmt(x, nd=1):
    return "樣本不足" if pd.isna(x) else f"{x:.{nd}f}"   # 不加千分位，避免和 CSV 的逗號分隔混淆

def build_knowledge_text():
    ev = df_supp
    sex = ev["sex"].where(ev["sex"].isin(["男", "女"]), "其他/未提供").value_counts(normalize=True)
    top_city = ev.loc[ev["city"] != "99_other", "city"].value_counts().head(5)
    by_year = ev["deliver"].dt.year.value_counts().sort_index()
    db1 = (
        f"bZ4X 車主補充資料（共 {len(ev):,} 位，資料截止日 {SNAPSHOT:%Y-%m-%d}）：\n"
        f"- 性別：{'、'.join(f'{k}{v:.0%}' for k, v in sex.items())}；年齡中位數 {ev['age'].median():.0f} 歲\n"
        f"- 居住縣市前五名：{'、'.join(f'{k}{v:,}人' for k, v in top_city.items())}\n"
        f"- 交車年度：{'、'.join(f'{int(k)}年{v:,}人' for k, v in by_year.items())}\n"
        f"- 近一年平均回廠 {_robust_mean(ev['visits_1y']):.1f} 次、平均維修 {_robust_mean(ev['repair_cnt_1y']):.1f} 次"
    )
    lines = [f"車款,動力,車主數,車齡中位數(年),車齡{lo}-{hi}年樣本數,車齡{lo}-{hi}年_年均定保花費(NT$),車齡{lo}-{hi}年_近一年平均回廠次數,車齡{lo}-{hi}年_近一年平均維修次數"]
    for m, r in MODEL_STATS.sort_values("owners", ascending=False).iterrows():
        lines.append(f"{m},{r['powertrain']},{r['owners']},{r['median_years']:.1f},{r['n_band']},"
                     f"{'樣本不足' if r['maint_fallback'] else _fmt(r['maint_per_year'], 0)},{_fmt(r['visits_band'])},{_fmt(r['repair_band'])}")
    db2 = "各車款車主回廠統計（主檔 {:,} 筆彙整）：\n".format(len(df_main)) + "\n".join(lines)
    return db1, db2

DB1_TEXT, DB2_TEXT = build_knowledge_text()

def _ask_gemini(system_instruction, prompt):
    last_err = None
    for name in GEMINI_MODELS:
        try:
            resp = client.models.generate_content(
                model=name, contents=prompt,
                config=types.GenerateContentConfig(system_instruction=system_instruction))
            return resp.text or "（顧問暫時沒有產生回覆，請換個問法再試一次）"
        except Exception as e:
            last_err = e
    raise last_err

# Gradio 6 起 Chatbot 只吃 messages 格式、theme 改放 launch()；舊版仍用 tuple → 這裡自動判斷
GR_MAJOR = int(gr.__version__.split(".")[0])
try:
    _CHATBOT_HAS_TYPE = "type" in inspect.signature(gr.Chatbot.__init__).parameters
except (TypeError, ValueError):
    _CHATBOT_HAS_TYPE = False
USE_MESSAGES = _CHATBOT_HAS_TYPE or GR_MAJOR >= 6

def get_ai_response(user_question, car_model, chat_history):
    if not user_question or not user_question.strip():
        return "", chat_history

    try:
        if not API_KEY or API_KEY == "YOUR_GEMINI_API_KEY_HERE":
            raise ValueError("尚未填入 Gemini API Key（區塊 0 的 GEMINI_API_KEY）")

        sys_instruction = f"""
        你是一位 TOYOTA 專屬的「EV LifePilot 油轉電 AI 隨行顧問」。
        你的任務是消除車主對電動車 (bZ4X) 的疑慮。
        請用具備同理心、親切且白話的比喻回答。

        請務必【嚴格根據以下兩份官方資料庫】來回答，引用數字時請說明是「車主回廠資料統計」。
        資料庫沒有的內容（例如電池保固年限、充電樁安裝規定與補助、車價）請誠實告知資料庫未提供，
        只給一般性的建議，並請客戶向 TOYOTA 經銷商確認，不要自行編造數字。
        金額原始單位為 A 國幣，本系統以 1 A國幣 = {A_CURRENCY_TO_NTD:g} 元新台幣換算後呈現。

        【資料庫 1】
        {DB1_TEXT}

        【資料庫 2】
        {DB2_TEXT}
        """
        cur = ""
        if car_model in ICE_STATS.index:
            r = ICE_STATS.loc[car_model]
            maint = "樣本不足" if r["maint_fallback"] else _fmt(r["maint_per_year"], 0)
            cur = f"（{car_model}：{r['powertrain']}，車主 {r['owners']} 位，車齡{lo}-{hi}年車主年均定保花費 {maint}）"
        prompt = f"【客戶目前駕駛車款】：{car_model}{cur}\n【客戶的疑問】：{user_question}"
        answer = _ask_gemini(sys_instruction, prompt)

    except Exception as e:
        answer = f"💡 顧問小提醒：發生異常，請確認 API Key 或資料庫狀態。({e})"

    chat_history = list(chat_history or [])
    if USE_MESSAGES:
        chat_history += [{"role": "user", "content": user_question},
                         {"role": "assistant", "content": answer}]
    else:
        chat_history.append((user_question, answer))
    return "", chat_history

In [25]:
# ---------------------------------------------------------
# 【區塊 4：UI 儀表板動態更新函數】
# ---------------------------------------------------------
def update_dashboard(car_type, annual_km):
    annual_km = int(annual_km)
    ice_cost, ev_cost, saved, d = calculate_tco(car_type, annual_km)

    fig = go.Figure(data=[
        go.Bar(name='現有燃油車', x=[f'{YEARS}年總花費(能源+保養)'], y=[ice_cost], marker_color='#E53935'),
        go.Bar(name='TOYOTA bZ4X', x=[f'{YEARS}年總花費(能源+保養)'], y=[ev_cost], marker_color='#4CAF50')
    ])

    fig.update_layout(
        title=f"<b>換購 bZ4X {YEARS} 年預估{'可為您省下' if saved >= 0 else '約多花'} NT$ {abs(saved):,.0f} 元</b>",
        barmode='group',
        height=380,
        margin=dict(l=20, r=20, t=50, b=20)
    )

    summary_text = (
        f"💡 **專屬診斷報告**：依您駕駛 **{car_type}**、每年行駛 **{annual_km:,}** 公里計算，"
        f"換購 bZ4X {YEARS} 年累積{'可省下' if saved >= 0 else '反而多花'}約 **NT$ {abs(saved):,.0f}** 元的油資與保養總支出！"
    )
    ice_src = (f"{car_type} 車齡{lo}-{hi}年樣本不足，採同為{d['powertrain']}車款的平均" if d["ice_fallback"]
               else f"{car_type} 車齡{lo}-{hi}年車主 {d['ice_n']:,} 位")
    summary_text += (
        f"\n\n📊 **資料依據（CSV）**：{ice_src}，每年定保平均 NT$ {d['ice_maint_yr']:,.0f}；"
        f"bZ4X 車主 {d['ev_n']:,} 位，每年定保平均 NT$ {d['ev_maint_yr']:,.0f}。"
        f"能源費用為假設值：{d['km_per_l']:g} km/L × 汽油 NT$ {GAS_PRICE_PER_L:g}/L；"
        f"{EV_KM_PER_KWH:g} km/kWh × 電價 NT$ {ELEC_PRICE_PER_KWH:g}/度。"
    )
    return fig, summary_text



In [26]:
# ---------------------------------------------------------
# 【區塊 5：建立 Gradio Web UI】（介面與原本完全相同，選單車款改由 CSV 產生）
# ---------------------------------------------------------
BLOCKS_KW = {} if GR_MAJOR >= 6 else {"theme": gr.themes.Soft()}
LAUNCH_KW = {"theme": gr.themes.Soft()} if GR_MAJOR >= 6 else {}

def build_app():
    with gr.Blocks(title="EV LifePilot", **BLOCKS_KW) as demo:
        gr.Markdown("# 🚗 TOYOTA EV LifePilot - 個人化油轉電 AI 隨行顧問")
        gr.Markdown("輸入您的用車習慣，AI 為您精算 5 年持有成本並解鎖專屬純電生活藍圖！")

        with gr.Row():
            with gr.Column(scale=1):
                car_input = gr.Dropdown(CAR_CHOICES, label="目前駕駛車款", value=CAR_CHOICES[0])
                km_input = gr.Slider(minimum=5000, maximum=50000, step=1000, label="預估年行駛里程 (公里)", value=15000)
                calc_btn = gr.Button("⚡ 立即精算 5 年 TCO 養車成本", variant="primary")
                output_text = gr.Markdown("點擊上方按鈕獲取診斷分析報告...")

            with gr.Column(scale=1):
                chart_output = gr.Plot(label="5 年持有成本比較圖")

        gr.Markdown("---")
        gr.Markdown("### 💬 24H 購車與充電疑慮諮詢 (GenAI 隨行顧問)")

        chatbot = gr.Chatbot(height=300, **({"type": "messages"} if _CHATBOT_HAS_TYPE else {}))
        msg_input = gr.Textbox(placeholder="您可以問我：大樓沒辦法裝充電樁怎麼辦？bZ4X 電池保固多久？", label="輸入您的疑問")

        calc_btn.click(fn=update_dashboard, inputs=[car_input, km_input], outputs=[chart_output, output_text])
        msg_input.submit(fn=get_ai_response, inputs=[msg_input, car_input, chatbot], outputs=[msg_input, chatbot])

    return demo

if __name__ == "__main__":
    demo = build_app()
    demo.launch(share=True, debug=True, **LAUNCH_KW)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://612c7dfc6e684b1083.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://612c7dfc6e684b1083.gradio.live
